### Document loader

In [7]:
from langchain_community.document_loaders import GitLoader

def file_filter(file_path: str) -> bool:
    return file_path.endswith(".mdx")

loader = GitLoader(
    clone_url="https://github.com/langchain-ai/langchain",
    repo_path="./langchain",
    branch="langchain==0.2.13",
    file_filter=file_filter
)

raw_docs = loader.load()
print(len(raw_docs))

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_32104\162678344.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import GitLoader


280


### Document transformer

In [8]:
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)

docs = text_splitter.split_documents(raw_docs)
print(len(docs))

Created a chunk of size 6803, which is longer than the specified 1000
Created a chunk of size 3302, which is longer than the specified 1000
Created a chunk of size 1851, which is longer than the specified 1000
Created a chunk of size 1639, which is longer than the specified 1000
Created a chunk of size 9269, which is longer than the specified 1000
Created a chunk of size 2579, which is longer than the specified 1000
Created a chunk of size 7107, which is longer than the specified 1000
Created a chunk of size 1819, which is longer than the specified 1000
Created a chunk of size 1548, which is longer than the specified 1000
Created a chunk of size 2047, which is longer than the specified 1000
Created a chunk of size 3304, which is longer than the specified 1000
Created a chunk of size 2867, which is longer than the specified 1000
Created a chunk of size 5726, which is longer than the specified 1000
Created a chunk of size 14093, which is longer than the specified 1000
Created a chunk of 

954


In [11]:
print(docs[0].page_content[:200])
print(docs[0].metadata)

# SQL Database Chain

This example demonstrates the use of the `SQLDatabaseChain` for answering questions over a SQL database.

Under the hood, LangChain uses SQLAlchemy to connect to SQL databases. T
{'source': 'cookbook\\sql_db_qa.mdx', 'file_path': 'cookbook\\sql_db_qa.mdx', 'file_name': 'sql_db_qa.mdx', 'file_type': '.mdx'}


### Embedding model

In [ ]:
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv

load_dotenv("../.env")

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

query = "AWS의 S3에서 데이터를 읽어 들이기 위한 Document loader가 있나요?"

vector = embeddings.embed_query(query)
print(len(vector))
print(vector)

In [9]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-base",
    # embed_documents()로 문서를 벡터화할 때 적용
    encode_kwargs={"prompt": "passage: ", "normalize_embeddings": True},
    # embed_query()로 검색 입력을 벡터화할 때 적용
    query_encode_kwargs={"prompt": "query: ", "normalize_embeddings": True}
    )

query = "AWS의 S3에서 데이터를 읽어 들이기 위한 Document loader가 있나요?"

vector = embeddings.embed_query(query)
print(len(vector))
print(vector)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1674.75it/s]


768
[0.021995745599269867, 0.0459619015455246, 0.023534635081887245, 0.013268555514514446, 0.028589311987161636, -0.01096186600625515, -0.020052116364240646, -0.04459156468510628, 0.006469614803791046, 0.0280288178473711, -0.035667914897203445, 0.02928859367966652, 0.14207403361797333, 0.05513635277748108, -0.043135832995176315, -0.07882779836654663, 0.029391853138804436, -0.027125200256705284, 0.03477463871240616, -0.02751723676919937, 0.08017413318157196, -0.019614819437265396, 0.07927586138248444, -0.02477838099002838, 0.028497369959950447, -0.0049025495536625385, -0.0012875132961198688, 0.012509778141975403, -0.03607659414410591, 0.056733306497335434, 0.023025980219244957, -0.05834052339196205, 0.019704554229974747, 0.020456157624721527, 0.006173921748995781, 0.051056861877441406, -0.02639651671051979, -0.020018434152007103, 0.014153112657368183, 0.01901303045451641, 0.019761523231863976, 0.013809737749397755, 0.026287034153938293, -0.015025955624878407, 0.026440368965268135, -0.01

In [10]:
# 청크로 분할한 문서와 Embedding model을 바탕으로 Vector store를 초기화
from langchain_chroma import Chroma

db = Chroma.from_documents(docs, embeddings)

In [13]:
# Vector store의 인스턴스에서 Retriever를 생성
retriever = db.as_retriever()

context_docs = retriever.invoke(query)
print(f"len: {len(context_docs)}\n")

first_doc = context_docs[0]
print(f"metatdata: {first_doc.metadata}\n")
print(first_doc.page_content)

len: 4

metatdata: {'file_name': 'aws.mdx', 'source': 'docs\\docs\\integrations\\platforms\\aws.mdx', 'file_type': '.mdx', 'file_path': 'docs\\docs\\integrations\\platforms\\aws.mdx'}

## Document loaders

### AWS S3 Directory and File

>[Amazon Simple Storage Service (Amazon S3)](https://docs.aws.amazon.com/AmazonS3/latest/userguide/using-folders.html)
> is an object storage service.
>[AWS S3 Directory](https://docs.aws.amazon.com/AmazonS3/latest/userguide/using-folders.html)
>[AWS S3 Buckets](https://docs.aws.amazon.com/AmazonS3/latest/userguide/UsingBucket.html)

See a [usage example for S3DirectoryLoader](/docs/integrations/document_loaders/aws_s3_directory).

See a [usage example for S3FileLoader](/docs/integrations/document_loaders/aws_s3_file).

```python
from langchain_community.document_loaders import S3DirectoryLoader, S3FileLoader
```

### Amazon Textract

>[Amazon Textract](https://docs.aws.amazon.com/managedservices/latest/userguide/textract.html) is a machine 
> learning 

### LCEL을 사용한 RAG Chain 구현

In [15]:
# 먼저 prompt(ChatPromptTemplate)와 model(ChatOpenAI)을 준비
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv("../.env")

prompt = ChatPromptTemplate.from_template('''\
다음 문맥만을 바탕으로 질문에 답변해 주세요.

문맥: """
{context}
"""

질문: {question}
''')

model = ChatOpenAI(model="gpt-6-luna")

In [16]:
# 이어서 LCEL로 RAG Chain을 구현하고 실행
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

output = chain.invoke(query)
print(output)

네. AWS S3에서 데이터를 읽기 위한 `S3DirectoryLoader`와 `S3FileLoader`가 있습니다. 각각 S3 디렉터리와 파일을 불러오는 데 사용하며, `langchain_community.document_loaders`에서 가져올 수 있습니다.


### 